In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import numpy as np
import time
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

block_size = 8
batch_size = 4

max_iterations = 10000
learning_rate = 3e-4
eval_iters = 250

cuda


In [2]:
with open('wizard_of_oz.txt', 'r', encoding='utf-8') as f:
    text = f.read()

chars = sorted(set(text))
print(chars)
print(len(chars))
vocab_size = len(chars)

['\n', ' ', '!', '"', '&', "'", '(', ')', '*', ',', '-', '.', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', ']', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
80


# Tokenizer

In [3]:
string_to_int = { c:i for i,c in enumerate(chars) }
int_to_string = { i:c for i,c in enumerate(chars) }
encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join([int_to_string[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)
print(data[:100])

tensor([ 1,  1, 28, 39, 42, 39, 44, 32, 49,  1, 25, 38, 28,  1, 44, 32, 29,  1,
        47, 33, 50, 25, 42, 28,  1, 33, 38,  1, 39, 50,  0,  0,  1,  1, 26, 49,
         0,  0,  1,  1, 36, 11,  1, 30, 42, 25, 38, 35,  1, 26, 25, 45, 37,  0,
         0,  1,  1, 25, 45, 44, 32, 39, 42,  1, 39, 30,  1, 44, 32, 29,  1, 47,
        33, 50, 25, 42, 28,  1, 39, 30,  1, 39, 50,  9,  1, 44, 32, 29,  1, 36,
        25, 38, 28,  1, 39, 30,  1, 39, 50,  9])


# Splits

In [4]:
n = int(0.8*len(data))

train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

x, y = get_batch("train")
print('inputs: ')
print(x)
print('targets: ')
print(y)

inputs: 
tensor([[ 1, 54, 67, 57,  0, 73, 61, 58],
        [58,  1, 72, 62, 65, 58, 67, 73],
        [62, 60, 65, 58, 73, 72, 24,  1],
        [57,  1, 78, 68, 74,  1, 58, 54]], device='cuda:0')
targets: 
tensor([[54, 67, 57,  0, 73, 61, 58, 71],
        [ 1, 72, 62, 65, 58, 67, 73,  9],
        [60, 65, 58, 73, 72, 24,  1, 49],
        [ 1, 78, 68, 74,  1, 58, 54, 73]], device='cuda:0')


In [5]:
x = train_data[:block_size]
y = train_data[1:block_size+1]

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print('When input is', context, 'target is', target)

When input is tensor([1]) target is tensor(1)
When input is tensor([1, 1]) target is tensor(28)
When input is tensor([ 1,  1, 28]) target is tensor(39)
When input is tensor([ 1,  1, 28, 39]) target is tensor(42)
When input is tensor([ 1,  1, 28, 39, 42]) target is tensor(39)
When input is tensor([ 1,  1, 28, 39, 42, 39]) target is tensor(44)
When input is tensor([ 1,  1, 28, 39, 42, 39, 44]) target is tensor(32)
When input is tensor([ 1,  1, 28, 39, 42, 39, 44, 32]) target is tensor(49)


# Model

In [6]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y) # calls forward
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [7]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, index, targets=None):
        logits = self.token_embedding_table(index)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss

    def generate(self, index, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self.forward(index)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            index_next = torch.multinomial(probs, num_samples=1)
            index = torch.cat((index, index_next), dim=1)
        
        return index

model = BigramLanguageModel(vocab_size)
m = model.to(device)

context = torch.zeros((1,1), dtype=torch.long, device = device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)


"Br&W!(6nQ04':
uk[&Ox.x(A_cSux,
*a7iy"5m]&y
H5).Tmx_0kOpMh,S1Z2Vhg1y366zx.8M".p xm]),bG3QxhwRg)J1Uy_rE9:h,&y0P36OGtd_EI0z.Rva3
!h,Vay02AB5HS3-XZS(f!43Rk.x.!vI_E(&K",wS".xp&VGd"egYASGNu7CAc2E7Q0

_1LQIUKGqd6H,n2?5x,(eo:GBZk6;L;*2&KnLDI7"ecVF6Y5p7BwSr,0(6q4
GNa_Co&VZnr7P6S:"g[&Q0nwK;HLh,2&QUwC:aBv1Y[.3;u.!BvBa156wzWUHLK1RzM:*UR4cF90aM3"8c9cuw5jv-?B GR.qLQ1NkKuUwBlx(AqD?Y2K'FdjFKHUT
sZ!HMH3xXky Tr7W??10-BM,2_EZAo!L6x;'d N_r7"JorY"ALHLW!E9w)!.!LRZ1'R'8h,NuBv&A
_n'pVRP9MAZy] Dj
Hvvz45gOw)U]u_:[pY!HHH


# Optimization

In [8]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iterations):
    if iter % eval_iters == 0:
        losses = estimate_loss()
        print(f"iteration {iter}, train loss: {losses['train']:.4f}, val loss: {losses['val']:.4f}")
    
    xb, yb = get_batch('train')

    logits, loss = model.forward(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())

iteration 0, train loss: 4.6609, val loss: 4.6496
iteration 250, train loss: 4.6116, val loss: 4.6036
iteration 500, train loss: 4.5287, val loss: 4.5277
iteration 750, train loss: 4.5058, val loss: 4.4820
iteration 1000, train loss: 4.4194, val loss: 4.4170
iteration 1250, train loss: 4.3750, val loss: 4.3592
iteration 1500, train loss: 4.3197, val loss: 4.3222
iteration 1750, train loss: 4.2611, val loss: 4.2534
iteration 2000, train loss: 4.2070, val loss: 4.1881
iteration 2250, train loss: 4.1587, val loss: 4.1384
iteration 2500, train loss: 4.0991, val loss: 4.0918
iteration 2750, train loss: 4.0486, val loss: 4.0571
iteration 3000, train loss: 4.0131, val loss: 4.0044
iteration 3250, train loss: 3.9430, val loss: 3.9588
iteration 3500, train loss: 3.9257, val loss: 3.9129
iteration 3750, train loss: 3.8582, val loss: 3.8799
iteration 4000, train loss: 3.8224, val loss: 3.8306
iteration 4250, train loss: 3.8024, val loss: 3.7893
iteration 4500, train loss: 3.7415, val loss: 3.7444

In [9]:
context = torch.zeros((1,1), dtype=torch.long, device = device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)



Sayheim;[KQx8pqD(Q7d nnqI4l5l, a
CRZ ieW!8pred,'8b4S J_rf[re ItatGd_GqDIotolt th(BopMM3QDS-al, t!FGRh;CounXBMk9d
wa[BFO2o5],2(A_t;
s annd bQw).
eKE&FGJo.)q_tGit, l, HM.QF g
EINR4JoW?Slk ocnn.iz.
_VmQ!Hisive a!lugoord;O8py sT2N4s im]DIAG;
Wq;1Liz3F0Ns4JKEY0p?C4


!t tond T ilutnong OQE
pZlWUhoo,?Qs BHM
kit I;ODe isL:9ds.Vis m]VxOLig.
gutut&hk?x!wlSO?8qgmsVserlif)g g I ig&WNd-t T&[:b N"L, sh*k

2P.!ll.UZP2.!-C
s"QcqDDiatcid.Vza6z)Uce
 RvecD aIald o:DqEse P1]B&GMGT

J-NQ_L7B y?8J3d wCyha0'?-
wBel5
